# EvolveAI — T4 GPU Autonomous Learning Starter

This notebook runs the existing custom SNN on a CUDA GPU, fetches only the allowlisted TinyStories dataset, and runs a **bounded, governed experiment cycle**. It saves experiment history and the best checkpoint to Google Drive.

**Important:** this is the first controlled prototype, not a conscious or superintelligent system. The current experiment selector searches reviewed configurations; it does not yet invent arbitrary learning rules. Internet access is restricted to the approved Hugging Face dataset path and the notebook does not execute downloaded code.

In [ ]:
from pathlib import Path
import subprocess, os, sys

REPO_DIR = Path('/content/EvolveAI')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/swasthik-s/EvolveAI.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

%cd /content/EvolveAI/snn
%pip -q install 'datasets>=3.0,<5.0'
print('Repository ready:', REPO_DIR)


In [ ]:
import torch
from train import resolve_device

device = resolve_device('auto')
print('PyTorch:', torch.__version__)
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
else:
    raise RuntimeError('CUDA GPU not detected. In Colab, choose Runtime > Change runtime type > T4 GPU, then rerun this cell.')


## 1. Connect persistent storage

Colab runtimes can reset. Mount Drive so the governor history and model checkpoint survive disconnections.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUN_DIR = Path('/content/drive/MyDrive/EvolveAI')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Persistent experiment folder:', RUN_DIR)


## 2. Acquire approved learning data

This streams at most 5 MiB from the allowlisted TinyStories dataset, deduplicates records, and creates a provenance manifest. Review the upstream dataset card and license before any use beyond research.

In [ ]:
import json
from dataset_harvester import fetch_corpus

CORPUS_PATH = RUN_DIR / 'tiny_stories_sample.txt'
MANIFEST_PATH = RUN_DIR / 'tiny_stories_manifest.json'

# Reuse the exact corpus from the previous cycle so the hash stays stable
# and compatible checkpoints can be continued safely.
if CORPUS_PATH.exists() and MANIFEST_PATH.exists():
    corpus = CORPUS_PATH.read_text(encoding='utf-8')
    manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    print('Reusing saved corpus from Drive.')
else:
    corpus, manifest = fetch_corpus('TinyStories (research starter)', max_mb=5.0)
    CORPUS_PATH.write_text(corpus, encoding='utf-8')
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print('Corpus bytes:', len(corpus.encode('utf-8')))
print('Records kept:', manifest['records_kept'])
print('SHA-256:', manifest['corpus_sha256'])
print('Dataset:', manifest['dataset_url'])
print('License note:', manifest['commercial_use_note'])


## 3. Run a governed autonomous experiment cycle

The governor checks dataset provenance, corpus size, trial count, step budget, and allowed architecture choices. It records every cycle and saves the best state dictionary. The model weights are not automatically deployed to the frontend.

In [ ]:
from governor import run_governed_cycle
from autonomous_lab import export_json

report = run_governed_cycle(
    corpus=corpus,
    manifest=manifest,
    state_path=RUN_DIR / 'governor_state.json',
    checkpoint_path=RUN_DIR / 'best_snn_checkpoint.pt',
    max_trials=4,
    steps=60,
    block_size=32,
    batch_size=8,
    seed=42,
    device='cuda',
)

(RUN_DIR / 'latest_experiment_report.json').write_text(export_json(report), encoding='utf-8')
print('Device used:', report['device_used'])
print('Learning focus:', report['learning_plan']['focus'])
print('Learning objective:', report['learning_plan']['objective'])
print('Hypothesis:', report['learning_plan']['hypothesis'])
print('Proposed experiments:', report['learning_plan']['proposed_experiments'])
print('Trials completed:', report['trials_completed'], '/', report['trials_requested'])
print('Best trial:', report['best_trial'])
print('Validation protocol:', report['validation_protocol'])
print('Elapsed seconds:', report['elapsed_seconds'])
print('Persistent outputs:', RUN_DIR)


In [ ]:
state = json.loads((RUN_DIR / 'governor_state.json').read_text(encoding='utf-8'))
print('Cycles recorded:', len(state['cycles']))
print('Historical best validation loss:', state.get('best_validation_loss'))
for cycle in state['cycles'][-10:]:
    print(cycle)


## What this first version does—and does not do

- **Does:** train on CUDA, stream approved data, run bounded experiments, save a best checkpoint, and retain experiment history in Drive.
- **Does not yet:** keep a Colab GPU process alive indefinitely, autonomously browse arbitrary internet pages, invent unrestricted code, or automatically replace the model serving the frontend.
- **Next milestone:** connect the Streamlit dashboard to a controlled training service that reads the same governor state, displays progress, and requires evaluation before a candidate checkpoint is promoted.

A Colab notebook is a development worker, not a reliable always-on API server. Do not expose the notebook runtime publicly with an unauthenticated tunnel.